# Category-reranked object RAG: design and plan

Question + category → mapped active documents across categories → eligible table/outer-row vectors → cosine candidate pool → category score reranking → exact cited evidence → Ollama code answer.

Retrieve up to four times `top_k` cosine matches, then rank by `score = (1 - cosine_distance) + CATEGORY_SCORE_BOOST * category_match`. The default bonus is `0.15`; configure `OBJECT_RAG_CATEGORY_BOOST` or `CATEGORY_SCORE_BOOST` below. Matching manual categories receive the bonus; other mapped categories remain eligible. Keep the raw distance alongside the similarity, applied bonus and final score for inspection. The constructor callback applies this ranking to direct `app.retrieve` and `app.answer` calls as well as the coverage suites. Saved outputs reflect earlier executions; restart the kernel, then rerun the configuration, service and selected retrieval/answer cells to refresh them.

Reuse the existing index without ingestion or checkpoint changes. Match the combined and V4.0/V5.0 workbook mappings through explicit corpus-root aliases, retain corpus and revision labels, and report missing coverage. Traverse row-group wrappers for outer rows; exclude cells and deeper rows from vector candidates. Validate identity, embedding configuration and dimensions; query eligible IDs before ranking.

Implementation: configure paths and services; load categories/mappings; open the catalog read-only and existing collection; inspect coverage; retrieve and inspect exact evidence; generate version-specific fenced snippets with citations; close resources. Defaults: supplied index, top-k 8, 24,000 context characters, Ollama RAG model falling back to the summary model. Complete blocks exceeding the budget are reported as omitted. Unsupported coding requests must receive an insufficient-evidence answer.

Validation: offline fake-provider and ephemeral-index tests, notebook validation and cell compilation, and read-only real catalog coverage inspection. No corpus rebuilds, summary requests or checkpoint resets.


Every answer includes a separate source-attributed summary of the retrieved objects, regardless of whether the coding request is supported.

## Application configuration

Reusable retrieval logic lives in [01.rag.object.ipynb](01.rag.object.ipynb), exported as `ribosome.rag.object`. Configuration and client lifecycle stay here.

In [ ]:
# | notest
import math
import os
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
from IPython.display import Markdown, display
from ribosome.rag.object import ObjectRAG
from ribosome.retrieval.contracts import IndexManifest, resolve_index_directory

PROJ_ROOT = Path(os.getenv("PROJ_ROOT", Path.cwd())).resolve()
while not (PROJ_ROOT / "pyproject.toml").exists() and PROJ_ROOT != PROJ_ROOT.parent:
    PROJ_ROOT = PROJ_ROOT.parent
load_dotenv(PROJ_ROOT / ".env", override=False)
EMBED_MODEL = os.getenv("OLLAMA_EMBED_MODEL", "qwen3-embedding:8b")
USE_TASKS = os.getenv("OBJECT_RAG_USE_TASKS", "1").lower() not in {"0", "false", "no"}
INDEX_DIR = resolve_index_directory(PROJ_ROOT, EMBED_MODEL, override=os.getenv("OBJECT_RAG_INDEX"), use_tasks=USE_TASKS)
INDEX_MANIFEST = IndexManifest.read(INDEX_DIR)
COLLECTION_NAME = INDEX_MANIFEST.collection if INDEX_MANIFEST else ("task_embeddings" if "task_embeddings" in INDEX_DIR.parts else "object_summaries")
RETRIEVAL_MODE = os.getenv("OBJECT_RAG_RETRIEVAL_MODE", "summary")  # hybrid is an explicit evaluated option.
HOSTS = sorted(set(x.strip().rstrip("/") for x in os.getenv("OLLAMA_EMBED_HOSTS", "http://localhost:11434,http://172.27.74.16:11434").split(",") if x.strip()))
RAG_MODEL = os.getenv("OLLAMA_RAG_MODEL") or os.getenv("OLLAMA_SUMMARY_MODEL", "")
RAG_HOST = os.getenv("OLLAMA_RAG_HOST") or os.getenv("OLLAMA_HOST", "http://localhost:11434")
MAX_CONTEXT_CHARS = 24000
REQUEST_TIMEOUT = 300
MAX_OUTPUT_TOKENS = 2048
CATEGORY_SCORE_BOOST = float(os.getenv("OBJECT_RAG_CATEGORY_BOOST", "0.15"))
if not math.isfinite(CATEGORY_SCORE_BOOST) or CATEGORY_SCORE_BOOST < 0:
    raise ValueError("OBJECT_RAG_CATEGORY_BOOST must be finite and nonnegative")

def rerank_by_category(hits, category):
    """Boost matching manual labels while retaining semantic similarity."""
    scored = []
    for hit in hits:
        similarity = 1.0 - hit["distance"]
        category_match = hit["manual"]["category"] == category
        boost = CATEGORY_SCORE_BOOST if category_match else 0.0
        scored.append(dict(hit, similarity=similarity, category_match=category_match,
                           category_boost=boost, score=similarity + boost))
    return sorted(scored, key=lambda hit: (-hit["score"], hit["distance"], hit["record"]["id"]))



## Open existing index and services
Run explicitly; no embedding or generation occurs in this cell.

In [ ]:
# | notest
import chromadb
from ollama import AsyncClient
if not (INDEX_DIR / 'catalog.sqlite3').is_file() or not (INDEX_DIR / 'chroma').is_dir():
    raise FileNotFoundError(INDEX_DIR)
headers = {'Authorization': 'Bearer '+os.environ['OLLAMA_API_KEY']} if os.getenv('OLLAMA_API_KEY') else {}
embedding_client = AsyncClient(host=HOSTS[0], headers=headers, timeout=REQUEST_TIMEOUT)
generation_client = AsyncClient(host=RAG_HOST, headers=headers, timeout=REQUEST_TIMEOUT)
collection = chromadb.PersistentClient(path=str(INDEX_DIR/'chroma')).get_collection(COLLECTION_NAME, embedding_function=None)
app = ObjectRAG(
    INDEX_DIR, collection, embedding_client, generation_client,
    mapping_dir=PROJ_ROOT / 'proj_config/coding_categories',
    embed_model=EMBED_MODEL,
    embedding_config=INDEX_MANIFEST.embedding_config if INDEX_MANIFEST else dict(provider='ollama-pool', hosts=HOSTS),
    retrieval_mode=RETRIEVAL_MODE,
    rag_model=RAG_MODEL, max_context_chars=MAX_CONTEXT_CHARS,
    max_output_tokens=MAX_OUTPUT_TOKENS, rerank=rerank_by_category,
)
async def retrieve(question, category, *, top_k=8):
    return await app.retrieve(question, category, top_k=top_k)
async def answer(question, category, *, top_k=8, mode="code"):
    return await app.answer(question, category, top_k=top_k, mode=mode)


## Category coverage

This suite has one prompt for each of the **17 application categories other than
`general_operation`**. Questions may use either mapped manual revision, with the
answer identifying its source. The dedicated manual-version suites and expanded CSV evaluation now live in
[06.test.rag.object.question_with_location.ipynb](06.test.rag.object.question_with_location.ipynb).

Set `COVERAGE_CASE_IDS` to selected category case IDs, or leave it `None` for all
17 prompts. The following two sections retrieve evidence and generate answers for
this suite only. Defining or displaying prompts makes no model requests. Shared
helpers also serve the separate manual suite, which has its own selection and
result dictionaries.


In [ ]:
# | notest
COVERAGE_CASES = [{'id': 'category_adhesive_dispensing',
  'category': 'adhesive_dispensing',
  'kind': 'category',
  'profile': '',
  'revision': '',
  'section': '',
  'section_title': '',
  'anchors': [],
  'question': '如何按涂胶手册配置胶机并编写一次涂胶流程，包含开始出胶、沿轨迹涂胶和停止出胶？请说明工艺文件和IO的准备条件。',
  'references': []},
 {'id': 'category_arc_welding',
  'category': 'arc_welding',
  'kind': 'category',
  'profile': '',
  'revision': '',
  'section': '',
  'section_title': '',
  'anchors': [],
  'question': '如何使用ARCON开始弧焊、沿焊缝运动并用ARCOFF结束焊接？请说明起弧文件与焊接文件的区别。',
  'references': []},
 {'id': 'category_fds',
  'category': 'fds',
  'kind': 'category',
  'profile': '',
  'revision': '',
  'section': '',
  'section_title': '',
  'anchors': [],
  'question': '如何按FDS手册配置主机工作参数和IO，并组织一次FDS作业的启动、完成状态确认及后续动作？',
  'references': []},
 {'id': 'category_grinding',
  'category': 'grinding',
  'kind': 'category',
  'profile': '',
  'revision': '',
  'section': '',
  'section_title': '',
  'anchors': [],
  'question': '如何首次配置打磨力控制器并完成标定，再设置打磨力、位移及出入刀监控，组织一次打磨作业？',
  'references': []},
 {'id': 'category_laser_welding',
  'category': 'laser_welding',
  'kind': 'category',
  'profile': '',
  'revision': '',
  'section': '',
  'section_title': '',
  'anchors': [],
  'question': '激光焊作业需要哪些激光器、通讯和IO配置？如何按手册组织激光开启、焊接运动和激光关闭的程序？',
  'references': []},
 {'id': 'category_load_analysis',
  'category': 'load_analysis',
  'kind': 'category',
  'profile': '',
  'revision': '',
  'section': '',
  'section_title': '',
  'anchors': [],
  'question': '使用负载分析软件时，如何确认工具数模坐标系与机器人法兰坐标系一致，并查看和解释负载分析结果？只需说明操作流程，不要虚构控制指令。',
  'references': []},
 {'id': 'category_material_handling',
  'category': 'material_handling',
  'kind': 'category',
  'profile': '',
  'revision': '',
  'section': '',
  'section_title': '',
  'anchors': [],
  'question': '如何配置搬运应用的抓手、工件传感器和电磁阀，并组织抓取工件、确认抓取状态、搬运后释放的作业？',
  'references': []},
 {'id': 'category_offline_interface',
  'category': 'offline_interface',
  'kind': 'category',
  'profile': '',
  'revision': '',
  'section': '',
  'section_title': '',
  'anchors': [],
  'question': '使用新松机器人离线接口库时，上位机初始化、建立网络连接、发送运动或下载作业、断开并释放资源的正确调用顺序是什么？请选择一种手册支持的语言举例。',
  'references': []},
 {'id': 'category_palletizing',
  'category': 'palletizing',
  'kind': 'category',
  'profile': '',
  'revision': '',
  'section': '',
  'section_title': '',
  'anchors': [],
  'question': '如何从开启码垛应用、PLC通讯准备、工具与用户坐标系设置开始，配置产品和码垛工艺，再组织一次码垛作业？',
  'references': []},
 {'id': 'category_pneumatic_spot_welding',
  'category': 'pneumatic_spot_welding',
  'kind': 'category',
  'profile': '',
  'revision': '',
  'section': '',
  'section_title': '',
  'anchors': [],
  'question': '如何配置气动点焊的设备交互IO、焊接压力和焊点参数，并组织加压、焊接、完成确认及松开焊钳的流程？',
  'references': []},
 {'id': 'category_spot_welding',
  'category': 'spot_welding',
  'kind': 'category',
  'profile': '',
  'revision': '',
  'section': '',
  'section_title': '',
  'anchors': [],
  'question': '伺服点焊作业前如何完成焊钳轴校零、压力标定和焊点参数配置，并组织一次加压、焊接及打开焊钳的流程？',
  'references': []},
 {'id': 'category_spr_riveting',
  'category': 'spr_riveting',
  'kind': 'category',
  'profile': '',
  'revision': '',
  'section': '',
  'section_title': '',
  'anchors': [],
  'question': '如何配置SPR铆机的工作模式、IO及通讯，并按基本作业流程完成一次铆接和防漏铆确认？',
  'references': []},
 {'id': 'category_stud_welding',
  'category': 'stud_welding',
  'kind': 'category',
  'profile': '',
  'revision': '',
  'section': '',
  'section_title': '',
  'anchors': [],
  'question': '如何配置螺柱焊焊机的工作参数、IO及通讯，并按基本作业流程完成一次焊接及防漏焊确认？',
  'references': []},
 {'id': 'category_teaching_free_welding',
  'category': 'teaching_free_welding',
  'kind': 'category',
  'profile': '',
  'revision': '',
  'section': '',
  'section_title': '',
  'anchors': [],
  'question': '免示教焊接软件如何配置机器人与视觉设备、创建工件、设置扫描区域和焊缝规则，再规划并执行焊接任务？请区分服务端配置和客户端操作。',
  'references': []},
 {'id': 'category_tool_quick_change',
  'category': 'tool_quick_change',
  'kind': 'category',
  'profile': '',
  'revision': '',
  'section': '',
  'section_title': '',
  'anchors': [],
  'question': '如何配置快换停靠站、快换盘和IO，并组织放下当前工具、确认解锁、拾取目标工具和确认锁紧的基本作业流程？',
  'references': []},
 {'id': 'category_tracking',
  'category': 'tracking',
  'kind': 'category',
  'profile': '',
  'revision': '',
  'section': '',
  'section_title': '',
  'anchors': [],
  'question': '如何使用FL_GETV读取第一路传送带的实时速度，并在跟随作业中使用该速度？请说明R变量、通路参数及单位。',
  'references': []},
 {'id': 'category_vision',
  'category': 'vision',
  'kind': 'category',
  'profile': '',
  'revision': '',
  'section': '',
  'section_title': '',
  'anchors': [],
  'question': '如何给相机发送拍照指令、获取测量结果并计算位姿偏移？请按同一手册版本说明视觉作业参数及指令调用顺序。',
  'references': []}]


In [ ]:
# | notest
import html
import re
from copy import deepcopy

def coverage_plain(text):
    return ' '.join(html.unescape(re.sub(r'<[^>]*>', ' ', text)).split())

def coverage_heading_title(heading):
    title = re.sub(r'^\s*\d+(?:\.\d+)*\s+', '', heading.strip('* '))
    return re.sub(r'\s+', '', title).translate(str.maketrans({'—': '-', '–': '-', '－': '-'}))

def coverage_heading_path_matches(expected_path, headings):
    remaining = iter(coverage_heading_title(heading) for heading in headings)
    return all(any(title == coverage_heading_title(expected) for title in remaining)
               for expected in expected_path)

def coverage_case_metrics(case, result, error=''):
    hits = result.get('hits', [])
    # Version-specific cases must find their own corpus and manual, even though
    # production retrieval is category-reranked and may return both revisions.
    matching = [hit for hit in hits if not case['profile'] or (
        hit['manual']['profile'] == case['profile'] and
        Path(hit['record']['source_file']).stem == Path(case['references'][0]['docx']).stem)]
    headings = [heading.split(maxsplit=1)[0] for hit in matching
                for heading in hit['record'].get('section_headings', []) if heading.strip()]
    if case.get('heading_path'):
        section_hit = any(coverage_heading_path_matches(
            case['heading_path'], hit['record'].get('section_headings', [])) for hit in matching)
    else:
        section_hit = case['section'] in headings if case['section'] else None
    source_text = '\n'.join(coverage_plain(hit['record']['content']) for hit in matching)
    anchors_found = all(re.search(r'(?<![A-Za-z0-9_])'+re.escape(term)+r'(?![A-Za-z0-9_])',
                                 source_text, flags=re.IGNORECASE) for term in case['anchors'])
    return dict(case_id=case['id'], category=case['category'], kind=case['kind'],
                expected_revision=case['revision'], expected_section=case['section'],
                hits=len(hits), expected_manual_hit=bool(matching) if case['profile'] else None,
                expected_section_hit=section_hit,
                anchors_in_matching_hits=anchors_found if case['anchors'] else None,
                context_chars=len(result.get('context', '')), omissions=len(result.get('omissions', [])),
                missing_manuals=len(result.get('coverage', {}).get('missing', [])), error=error)

def show_coverage_retrieval(case_id, *, cases=None, results=None, errors=None):
    cases = coverage_cases if cases is None else cases
    results = coverage_retrieval_results if results is None else results
    errors = coverage_retrieval_errors if errors is None else errors
    case = next(case for case in cases if case['id'] == case_id)
    display(Markdown(f"### {case_id}\n\n{case['question']}"))
    if case_id in errors:
        print(errors[case_id])
        return
    result = results[case_id]
    display(pd.DataFrame([dict(score=hit.get('score'), similarity=hit.get('similarity'),
                               category_match=hit.get('category_match'),
                               category_boost=hit.get('category_boost'),
                               distance=hit['distance'], manual=hit['manual'],
                               lines=(hit['record']['attributes'].get('line_start'),
                                      hit['record']['attributes'].get('line_end')),
                               summary=hit['record'].get('summary', ''),
                               source=hit['record']['source_file']) for hit in result['hits']]))
    print(result['context'])
    print('Omitted:', result['omissions'])
    # Reference excerpts remain reviewer data and are never added to the prompt.
    if case['references']:
        display(pd.DataFrame(case['references']))

class CoverageAnswerReplay(ObjectRAG):
    """Run the production answer method on the exact retrieved test evidence."""
    def __init__(self, application, evidence, *, top_k=8):
        self.generator = application.generator
        self.rag_model = application.rag_model
        self.max_output_tokens = application.max_output_tokens
        self.evidence = evidence
        self.top_k = top_k
        self.last_result = None

    async def retrieve(self, question, category, *, top_k=8):
        if (question, category, top_k) != (
            self.evidence['question'], self.evidence['category'], self.top_k
        ):
            raise ValueError('Coverage replay requires the same question, category and top-k')
        self.last_result = deepcopy(self.evidence)
        return self.last_result

async def run_coverage_retrievals(application, cases, *, top_k, results, errors):
    """Retrieve into this suite's dictionaries; retain completed cases on interruption."""
    for number, case in enumerate(cases, 1):
        print(f"[{number}/{len(cases)}] Retrieve {case['id']}", flush=True)
        try:
            results[case['id']] = await application.retrieve(
                case['question'], case['category'], top_k=top_k)
        except Exception as error:
            errors[case['id']] = f'{type(error).__name__}: {error}'
    return pd.DataFrame([
        coverage_case_metrics(case, results.get(case['id'], {}),
                              errors.get(case['id'], ''))
        for case in cases])

async def run_coverage_answers(application, cases, *, top_k, retrieval_results,
                               retrieval_errors, results, errors):
    """Answer from this suite's frozen evidence; never make another retrieval request."""
    for number, case in enumerate(cases, 1):
        case_id = case['id']
        display(Markdown(f"## [{number}/{len(cases)}] {case_id}\n\n{case['question']}"))
        if case_id not in retrieval_results:
            errors[case_id] = retrieval_errors.get(case_id, 'Retrieve this case first')
            display(Markdown('Retrieval failed: ' + errors[case_id]))
            continue
        replay = CoverageAnswerReplay(application, retrieval_results[case_id], top_k=top_k)
        try:
            results[case_id] = await replay.answer(
                case['question'], case['category'], top_k=top_k)
        except Exception as error:
            errors[case_id] = f'{type(error).__name__}: {error}'
            results[case_id] = replay.last_result or {}
        finally:
            if case_id not in results and replay.last_result is not None:
                results[case_id] = replay.last_result
                errors[case_id] = 'Generation interrupted before completion'
            case_result = results.get(case_id, {})
            if 'retrieved_summary' in case_result:
                display(Markdown('### Retrieved content summary\n\n' + case_result['retrieved_summary']))
        if case_id in errors:
            display(Markdown('Generation failed: ' + errors[case_id]))
        else:
            display(Markdown('### Coding or diagnostic answer\n\n' + case_result['answer']))
    return pd.DataFrame([
        dict(**coverage_case_metrics(case, results.get(case['id'], {}),
                                     errors.get(case['id'], '')),
             example_status=results.get(case['id'], {}).get('example_status', 'not generated'))
        for case in cases])


In [ ]:
# | notest
display(pd.DataFrame([(category, name) for category, name in app.categories.items()
                      if category != 'general_operation'], columns=['category', 'chinese_name']))
assert {case['category'] for case in COVERAGE_CASES} == set(app.categories) - {'general_operation'}
COVERAGE_TOP_K = 8
COVERAGE_CASE_IDS = None  # Example: ['category_arc_welding', 'category_vision']
known_case_ids = {case['id'] for case in COVERAGE_CASES}
if COVERAGE_CASE_IDS is not None:
    if not COVERAGE_CASE_IDS or set(COVERAGE_CASE_IDS) - known_case_ids:
        raise ValueError('Select at least one known coverage case ID')
coverage_cases = [case for case in COVERAGE_CASES
                  if COVERAGE_CASE_IDS is None or case['id'] in COVERAGE_CASE_IDS]
with pd.option_context('display.max_rows', None, 'display.max_colwidth', None):
    display(pd.DataFrame(coverage_cases)[
        ['id', 'category', 'question']])
category_coverage_rows = []
for category in dict.fromkeys(case['category'] for case in coverage_cases):
    mapped_coverage = app.snapshot(category)[2]
    category_coverage_rows.append(dict(category=category, indexed=mapped_coverage['indexed'],
                                       missing=mapped_coverage['missing']))
display(pd.DataFrame(category_coverage_rows))


In [ ]:
# | notest
INSPECT_CASE_ID = coverage_cases[0]['id']  # Change to any selected ID to inspect its exact evidence.


## Retrieve and inspect exact evidence

Run the selected category prompts through the existing category-reranked retriever.
Results are retained by case ID in `coverage_retrieval_results`, with per-case
failures in `coverage_retrieval_errors`; completed results remain inspectable after
interruption. Use `show_coverage_retrieval(case_id)` for any selected category case
to inspect the matched objects, raw context and omissions.

The category table reports hit count, context size, omitted objects, missing mapped
manuals and errors. These are retrieval diagnostics, not answer-correctness scores.
Version, subsection and alarm-code checks belong to the dedicated manual suite
below. The original production candidate restrictions still apply.


In [ ]:
# | notest
coverage_retrieval_results, coverage_retrieval_errors = {}, {}
coverage_retrieval_metrics = await run_coverage_retrievals(
    app, coverage_cases, top_k=COVERAGE_TOP_K,
    results=coverage_retrieval_results, errors=coverage_retrieval_errors)
display(coverage_retrieval_metrics[
    ['case_id', 'category', 'hits', 'context_chars', 'omissions', 'missing_manuals', 'error']])
show_coverage_retrieval(INSPECT_CASE_ID)


## Retrieved content summary and coding answer

Always display a source-attributed summary of the found objects, even when their content cannot support the requested code. The summary uses stored object summaries and remains available if generation fails. With no hits, explicitly report that no relevant indexed content was found.

The coding answer ends with a meaningful fenced code snippet for the selected category, integrating the instruction requested in the question. Its source citations, corpus/revision, and required placeholders appear directly before the code block. If the evidence cannot support such an example, the answer explains the missing information.

A rich example includes a concrete scenario, environment and initial values, expected outcome, and a final multi-statement code block with setup, the requested instruction, and follow-up. The shared RAG class validates structured generation and retries once if incomplete. If evidence cannot support the surrounding code, it explains the missing context. After updating the shared module, restart the kernel and rerun configuration and client setup to load the new implementation.

Each selected case reuses the **exact evidence from the preceding retrieval pass**
through the production answer method, avoiding a second embedding request or a
different retrieval result. It displays a retrieved-content summary and answer for
every case; generation failures retain the available summary. Completed answers
remain in `coverage_answer_results` if execution is interrupted.

The coding-answer validator may return `unsupported` for GUI or workflow
questions, which is separate from retrieval coverage. Review the explanation and
sources. A structurally `included` example still requires source checking.


In [ ]:
# | notest
coverage_answer_results, coverage_answer_errors = {}, {}
coverage_answer_metrics = await run_coverage_answers(
    app, coverage_cases, top_k=COVERAGE_TOP_K,
    retrieval_results=coverage_retrieval_results, retrieval_errors=coverage_retrieval_errors,
    results=coverage_answer_results, errors=coverage_answer_errors)
display(coverage_answer_metrics[
    ['case_id', 'category', 'hits', 'context_chars', 'omissions', 'missing_manuals', 'example_status', 'error']])


## Cleanup
Run after the selected category tests, or after interruption.


In [ ]:
# | notest
try:
    if 'app' in globals(): app.close()
finally:
    for client_name in ('embedding_client', 'generation_client'):
        if client_name in globals(): await globals()[client_name]._client.aclose()